# QOMM — reproducing the experiments and drawing them

This notebook **does not re-run any measurement**. It reads what is in
`artifacts/` and prints the numbers and draws the figures. The measuring is
done by the `make` targets, which leave their results as JSON. This is the reading side.

Why they are separate: some measurements need seven-node secure computation or a full
Ethereum archive node, and will not run everywhere. Keeping the reading side apart means
anyone can check whatever measurements they do have.

| experiment | command | output |
|---|---|---|
| cost of settlement | `make defmi` | `defmi.json` |
| the Rust port | `make rust-bench` | `rust_bench.json` |
| sweeping the attacker's power | `make rho-sweep` | `rho_sweep.json` |
| eight real instruments | `make sim-real` | `sim_matrix_bybit.json` |
| a real RFQ venue | `make sim-rfq` | `sim_matrix_uniswapx.json` |
| correcting DP disclosure | `make dp-effect` | `dp_effect.json` |
| the fixed cost of a quote | `make mpc-resident` | `mpc_resident.json` |
| multiple relay hops | `make transport` | `transport.json` |
| WebAssembly | `make wasm-bench` | `wasm_native.json`, `wasm_wasm32.json` |
| auditing inventory updates | `make state-audit` | `state_audit.json` |
| where the nodes sit | `make placement` | `placement.json` |

In [ ]:
import json, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ART = ROOT / "artifacts"
sys.path.insert(0, str(ROOT))

def load(name):
    path = ART / name
    if not path.exists():
        print(f"{name} is not here; run the matching make target first")
        return None
    if name.endswith(".jsonl"):
        return [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    return json.loads(path.read_text())

print("present:", ", ".join(sorted(p.name for p in ART.glob("*.json"))[:8]), "...")

## 0. Redraw every figure

Every figure is drawn from the measurements by `scripts/make_figures.py`.
A figure with no measurement behind it is skipped, with the reason given.

In [ ]:
!cd {ROOT} && python3 scripts/make_figures.py

## 1. The headline result: of the two numbers compared, only one is about the protocol

How well an unsettled request can be guessed, swept over how much of the
wallet ownership the attacker already knows.

- **This proposal sits at exactly 0.500 at every fraction** --- which is what an empty clue arithmetically gives
- The ordinary scheme's value is set almost entirely by that fraction

So *how large* the gap is depends on what is assumed about the attacker;
what is about the protocol is that it does not depend on it at all.

In [ ]:
data = load("rho_sweep.json")
if data:
    for arm, payload in data["arms"].items():
        print(f"\n{payload['meta'].get('source', arm)}")
        print(f"  {'fraction':>9} {'this design':>12} {'plain RFQ':>10}")
        rhos = sorted({r["linkage_rho"] for r in payload["rows"]})
        for rho in rhos:
            pick = lambda p: next((r["auc_mean"] for r in payload["rows"]
                                   if r["protocol"] == p and r["linkage_rho"] == rho), None)
            q, pl = pick("qomm_rfq"), pick("plain_rfq")
            print(f"  {rho:>6} {q:>10.4f} {pl:>10.4f}")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ART / "figures" / "rho_sweep.png")))

## 2. Eight real instruments: unchanged across a 91x span of liquidity

Eight perpetual futures (0.18 to 15.93 arrivals a second): 6 protocols x 3 disclosures x 2 layers x 5 seeds, 1,440 runs.

In [ ]:
import re
data = load("sim_matrix_bybit.json")
if data:
    rows = [r for r in (data.get("aggregate") or []) if "source" in r]
    raw = data.get("rows") or []
    sym = lambda s: (re.search(r"bybit:([A-Z0-9]+)", s or "") or [None, "?"])[1]
    rate = {sym((r.get("tape") or {}).get("source")): (r.get("tape") or {}).get("arrival_per_s", 0)
            for r in raw if r.get("tape")}
    val = lambda r, k: (r.get(k) or {}).get("mean") if isinstance(r.get(k), dict) else r.get(k)
    print(f"  {'symbol':>9} {'per s':>7} {'this design':>12} {'plain RFQ':>10}")
    for s in sorted(rate, key=lambda x: -rate[x]):
        cells = {r["protocol"]: val(r, "A1_passive_observer.auc") for r in rows
                 if sym(r["source"]) == s and r["disclosure"] == "A_none"
                 and r["layer"] == "replay"}
        f = lambda p: f"{cells[p]:.4f}" if cells.get(p) is not None else "undefined"
        print(f"  {s:>9} {rate[s]:>7.2f} {f('qomm_rfq'):>10} {f('plain_rfq'):>10}")
    print("\nThe thickest symbol is undefined because every entity settles somewhere every time,")
    print("so nothing is left that asked and settled nothing --- the thing being measured.")

In [ ]:
display(Image(filename=str(ART / "figures" / "real_market.png")))

## 3. Settlement cost follows the ledger's width, and what the Rust port changed

It is not only the language being compared. The port also swapped the range
proof for an audited one, so the difference below is the two changes added together.

In [ ]:
py, rs = load("defmi.json"), load("rust_bench.json")
if py and rs:
    by_bits = {r["bits"]: r for r in rs["scaling"]}
    print(f"  {'bits':>6} {'Python':>10} {'Rust':>10} {'ratio':>7} {'Py package':>12} {'Rust package':>13}")
    for r in py["scaling"]:
        o = by_bits.get(r["bits"])
        if not o:
            continue
        print(f"  {r['bits']:>6} {r['settle_ms']:>9.1f}ms {o['settle_ms']:>9.2f}ms "
              f"{r['settle_ms']/o['settle_ms']:>6.1f}x {r['package_bytes']:>10,}B {o['package_bytes']:>11,}B")
    print(f"\ncalibration: Python {py['calibration']['scalar_mult_us']:.1f} us / "
          f"Rust {rs['calibration']['scalar_mult_us']:.1f} us")
    print("If these two disagree, the ratios above may be the machine and not the code.")

In [ ]:
display(Image(filename=str(ART / "figures" / "settlement_cost.png")))

## 4. Correcting DP disclosure: it works on a synthetic market and vanishes on real data

Before and after fixing two defects --- upward bias from an absolute value, and a
mismatched basis for the noise --- paired on the same run and the same seed.

In [ ]:
data = load("dp_effect.json")
if data:
    for arm, payload in data["arms"].items():
        print(f"\n{payload['meta'].get('source', arm)} (difference against no disclosure)")
        for kind, metrics in payload["paired_against_no_disclosure"].items():
            label = "as published" if kind == "dp_uncorrected" else "corrected"
            for metric, stat in metrics.items():
                if stat["mean"] is None:
                    continue
                name = "fill rate" if metric == "fill_rate" else "maker P&L/fill"
                mark = "significant" if stat["excludes_zero"] else "not significant"
                print(f"  {label:>8} {name:>12} {stat['mean']:+9.4f} "
                      f"± {stat['half_width']:.4f}  ({mark})")

In [ ]:
display(Image(filename=str(ART / "figures" / "dp_effect.png")))

## 5. The fixed cost of a quote: what worked was batching, not keeping processes resident

The comparison arm (not resident) is measured in the same run, so the two
effects can be told apart.

In [ ]:
data = load("mpc_resident.json")
if data:
    cold = {r["batch"]: r for r in data["cold"]}
    warm = {r["batch"]: r for r in data["resident"]}
    print(f"  {'batch':>8} {'cold':>10} {'resident':>10} {'residency gain':>15}")
    for b in sorted(cold):
        print(f"  {b:>10} {cold[b]['ms_per_quote']:>9.1f}ms {warm[b]['ms_per_quote']:>9.1f}ms "
              f"{cold[b]['ms_per_quote']/warm[b]['ms_per_quote']:>10.2f}x")
    lo, hi = min(cold), max(cold)
    print(f"\n  batching alone (cold, {lo} to {hi}): {cold[lo]['ms_per_quote']/cold[hi]['ms_per_quote']:.2f}x")
    print(f"  residency alone (at {lo}):            {cold[lo]['ms_per_quote']/warm[lo]['ms_per_quote']:.2f}x")

In [ ]:
display(Image(filename=str(ART / "figures" / "mpc_residency.png")))

## 6. The remaining figures

Only the ones with a measurement behind them are shown.

In [ ]:
for stem, caption in [("parallel_scaling", "workers against throughput"),
                      ("relay_hops", "relay hops, each with its own clock"),
                      ("wasm_vs_native", "WebAssembly against native"),
                      ("anonymity_set", "the anonymity set is paid for by the payer"),
                      ("state_audit", "auditing inventory does not grow with history"),
                      ("node_placement", "one distant node is as bad as all of them")]:
    path = ART / "figures" / f"{stem}.png"
    if path.exists():
        print(caption)
        display(Image(filename=str(path)))
    else:
        print(f"{caption}: {stem}.png is not here")

## 7. Checking the numbers can be trusted

This project once compared numbers across a change in the machine's state without noticing.
Every measurement since carries a **calibration**: the time for one elementary operation.
Compare that before comparing anything else.

In [ ]:
for name in ("defmi.json", "rust_bench.json", "state_audit.json",
             "mpc_resident.json", "wasm_native.json", "defmi_host_a.json"):
    data = load(name)
    if data and "calibration" in data:
        print(f"  {name:24} {data.get('host','?'):>8}  "
              f"scalar mult {data['calibration']['scalar_mult_us']:.2f} us")